<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# E-Commerce Analytics 360
## Étape 3 — SQL Analytics, KPIs & Segmentation RFM
### ✅ VERSION CORRIGÉE

> **Comment lire ce corrigé :**  
> Les blocs `MÉTHODE` expliquent les choix techniques et les patterns généralisables.  
> Les blocs `INTERPRÉTATION` lisent les résultats pour M. Diallo.  
> Les blocs `MÉTIER` font le lien entre le chiffre et la décision business.

| | |
|---|---|
| **Niveau** | Avancé |
| **Outils** | Python, DuckDB (JupySQL), Matplotlib |
| **Durée estimée** | 2h à 3h |

---
## 0. Mise en place de l'environnement

In [ ]:
!pip install jupysql duckdb-engine --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.2f}".format)

COLORS = {
    "primary":   "#534AB7",
    "secondary": "#1D9E75",
    "warning":   "#EF9F27",
    "danger":    "#E24B4A",
    "neutral":   "#888780",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F9F9F8",
    "axes.grid":        True,
    "grid.alpha":       0.35,
    "font.size":        11,
})

print("Environnement prêt ✅")

---
## 1. Chargement des données et connexion DuckDB

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/data/"

fact_ec = pd.read_csv(BASE_URL + "fact_ecommerce_analytics.csv", parse_dates=["order_date"])

print(f"Dataset chargé       : {len(fact_ec):,} lignes")
print(f"Commandes livrées    : {fact_ec[fact_ec['order_status']=='Livree']['order_id'].nunique():,}")
print(f"Période              : {fact_ec['order_date'].min().date()} → {fact_ec['order_date'].max().date()}")

In [ ]:
%load_ext sql
%sql duckdb:///:memory:

# Enregistrer le DataFrame pandas comme table DuckDB
%sql --persist fact_ec

print("Connexion DuckDB prête ✅")
print("Table disponible : fact_ec")

> **MÉTHODE — SQL vs Python : choisir le bon outil pour chaque usage.**
>
> **SQL est supérieur pour :** les agrégations sur de grands volumes (GROUP BY, SUM, COUNT), les jointures, le partage des KPIs avec d'autres équipes, la reproductibilité.
>
> **Python est supérieur pour :** les opérations sur les dataframes (`pd.qcut`, `apply`), la visualisation avancée, le machine learning.
>
> **Le workflow pro :** SQL extrait et agrège → Python analyse et visualise.
>
> **Syntaxe DuckDB vs SQL Server :**
> - `FORMAT(order_date, 'yyyy-MM')` → `strftime(order_date, '%Y-%m')`
> - `TOP N` → `LIMIT N`
> - Window functions, CTEs, HAVING, CASE WHEN : **identiques**

---
# PARTIE 1 — SQL Analytics avec DuckDB

> **Rappel fondamental : toujours filtrer sur `order_status = 'Livree'` pour les KPIs financiers.**  
> Seule exception : les analyses de qualité opérationnelle (taux d'annulation) où on veut délibérément compter toutes les commandes.

---
## Bloc A — KPIs Fondamentaux

> **MÉTHODE — `SUM(margin) / SUM(revenue)` et non `AVG(margin / revenue)`.**
>
> Ces deux calculs ne donnent pas le même résultat. `SUM(margin) / SUM(revenue)` : taux de marge sur le CA total (pondération par volume). C'est la métrique correcte pour un taux de marge global.

### A1 — KPIs globaux

In [ ]:
%%sql
SELECT
    ROUND(SUM(revenue), 0)                          AS ca_total,
    ROUND(SUM(margin), 0)                           AS marge_totale,
    ROUND(SUM(margin) / SUM(revenue) * 100, 1)      AS taux_marge_pct,
    COUNT(DISTINCT order_id)                         AS nb_commandes
FROM fact_ec
WHERE order_status = 'Livree'

> **INTERPRÉTATION — Taux de marge global (~37%) :** pour chaque 100€ vendus, 37€ restent après le coût d'achat. En e-commerce multi-catégories, 30-40% est standard. Si < 25% : problème de mix produit. Si > 45% : positionnement premium.

### A2 — Panier moyen (requête en deux étapes)

> **MÉTHODE — Pourquoi deux étapes et non `AVG(revenue)` directement ?**
>
> `AVG(revenue)` calcule la moyenne par ligne. Une commande avec 3 produits a 3 lignes. Le **panier moyen** = montant total par commande, pas par ligne produit. Pattern : **agréger puis agréger**.

In [ ]:
%%sql
SELECT ROUND(AVG(montant_commande), 2) AS panier_moyen
FROM (
    SELECT order_id, SUM(revenue) AS montant_commande
    FROM fact_ec
    WHERE order_status = 'Livree'
    GROUP BY order_id
) AS commandes_agreg

### A3 — Clients actifs

> **MÉTHODE — `COUNT(DISTINCT customer_id)` et non `COUNT(customer_id)`.**  
> Un client avec 5 commandes a 5 lignes dans `fact_ec`. `COUNT` retournerait 5. `COUNT(DISTINCT)` retourne 1. Règle : pour compter des entités uniques, toujours `COUNT(DISTINCT clé_primaire)`.

In [ ]:
%%sql
SELECT COUNT(DISTINCT customer_id) AS nb_clients_actifs
FROM fact_ec
WHERE order_status = 'Livree'

---
## Bloc B — Analyse Temporelle

> **MÉTHODE — L'analyse temporelle répond à 'comment on évolue ?' et non juste 'où en est-on ?'**  
> Un CA de 7.2M€ sur 30 mois peut masquer une tendance catastrophique si les 6 derniers mois sont en forte baisse.

### B4 — CA et marge mensuels

> **DuckDB : `strftime(order_date, '%Y-%m')` remplace `FORMAT(order_date, 'yyyy-MM')` de SQL Server.**

In [ ]:
%%sql
SELECT
    strftime(order_date, '%Y-%m')                   AS mois,
    ROUND(SUM(revenue), 0)                          AS ca_mensuel,
    ROUND(SUM(margin), 0)                           AS marge_mensuelle,
    ROUND(SUM(margin) / SUM(revenue) * 100, 1)      AS taux_marge_pct,
    COUNT(DISTINCT order_id)                         AS nb_commandes
FROM fact_ec
WHERE order_status = 'Livree'
GROUP BY strftime(order_date, '%Y-%m')
ORDER BY mois

> **INTERPRÉTATION — Un taux de marge qui varie fortement selon les mois** signale un changement de mix produit. Les mois de soldes ont souvent un taux de marge plus bas (produits promos à prix réduit). Si le taux baisse même hors promotion : soit les coûts d'achat ont augmenté, soit le mix glisse vers des catégories moins rentables.

### B5 — Top 3 mois par CA

> **DuckDB : `LIMIT N` remplace `TOP N` de SQL Server. `LIMIT` s'écrit en fin de requête, après `ORDER BY`.**

In [ ]:
%%sql
SELECT
    strftime(order_date, '%Y-%m')   AS mois,
    ROUND(SUM(revenue), 0)          AS ca
FROM fact_ec
WHERE order_status = 'Livree'
GROUP BY strftime(order_date, '%Y-%m')
ORDER BY ca DESC
LIMIT 3

### B6 — Anomalies : CA en hausse mais marge en baisse ⭐ (LAG)

> **MÉTHODE — `LAG(ca) OVER (ORDER BY mois)` : la window function de comparaison temporelle.**
>
> `LAG(ca) OVER (ORDER BY mois)` retourne la valeur de la ligne précédente dans l'ordre chronologique. C'est une **window function** : elle ne groupe pas les lignes, elle les compare entre elles.
>
> **Pourquoi deux CTEs ?** On ne peut pas filtrer sur le résultat d'une window function dans la même CTE. La première CTE calcule les deltas, la seconde filtre sur ces deltas.

In [ ]:
%%sql
WITH monthly AS (
    SELECT
        strftime(order_date, '%Y-%m')   AS mois,
        SUM(revenue)                    AS ca,
        SUM(margin)                     AS marge
    FROM fact_ec
    WHERE order_status = 'Livree'
    GROUP BY strftime(order_date, '%Y-%m')
),
monthly_delta AS (
    SELECT
        mois,
        ROUND(ca, 0)                                            AS ca,
        ROUND(marge, 0)                                         AS marge,
        ROUND(ca    - LAG(ca)    OVER (ORDER BY mois), 0)      AS delta_ca,
        ROUND(marge - LAG(marge) OVER (ORDER BY mois), 0)      AS delta_marge
    FROM monthly
)
SELECT *
FROM monthly_delta
WHERE delta_ca > 0
  AND delta_marge < 0
ORDER BY mois

> **MÉTIER — Un mois où le CA monte mais la marge baisse** signale que la croissance vient de produits moins rentables (promotions, catégories à faible marge). Ce sont des mois à investiguer en priorité pour comprendre si le mix produit se dégrade structurellement.

---
## Bloc C — Produits & Catégories

### C7 — Performance par catégorie avec double ranking ⭐

> **MÉTHODE — `RANK() OVER()` : classer sans filtrer.**
>
> `RANK()` attribue un rang à chaque ligne sans supprimer les autres lignes. On voit le rang de **toutes** les catégories simultanément — impossible avec `ORDER BY + LIMIT`.
>
> **RANK vs DENSE_RANK vs ROW_NUMBER :**  
> `RANK()` : 1, 2, 2, 4 (saute le rang 3 si égalité)  
> `DENSE_RANK()` : 1, 2, 2, 3 (ne saute pas)  
> `ROW_NUMBER()` : 1, 2, 3, 4 (toujours unique)

In [ ]:
%%sql
SELECT
    categorie,
    ROUND(SUM(revenue), 0)                          AS ca_total,
    ROUND(SUM(margin), 0)                           AS marge_totale,
    ROUND(SUM(margin) / SUM(revenue) * 100, 1)      AS taux_marge_pct,
    SUM(quantite)                                   AS qte_vendue,
    COUNT(DISTINCT order_id)                         AS nb_commandes,
    RANK() OVER (ORDER BY SUM(revenue) DESC)        AS rang_ca,
    RANK() OVER (ORDER BY SUM(margin)  DESC)        AS rang_marge
FROM fact_ec
WHERE order_status = 'Livree'
GROUP BY categorie
ORDER BY ca_total DESC

> **INTERPRÉTATION — La divergence rang CA vs rang marge :**  
> Une catégorie classée 1 en CA mais 4 en marge est une catégorie de volume peu rentable. Une catégorie classée 4 en CA mais 1 en marge est une perle cachée. **Action :** réallouer le budget marketing des catégories de volume vers les catégories de marge.

### C8 — Top 10 marques par quantité vendue

In [ ]:
%%sql
SELECT
    marque,
    categorie,
    SUM(quantite)                               AS qte_vendue,
    ROUND(SUM(revenue), 0)                      AS ca,
    ROUND(SUM(margin) / SUM(revenue) * 100, 1)  AS taux_marge_pct
FROM fact_ec
WHERE order_status = 'Livree'
GROUP BY marque, categorie
ORDER BY qte_vendue DESC
LIMIT 10

---
## Bloc D — Analyse Clients

### D9 — Top 20 clients avec double ranking

In [ ]:
%%sql
SELECT
    customer_id, pays, segment_client,
    COUNT(DISTINCT order_id)                            AS nb_commandes,
    ROUND(SUM(revenue), 0)                              AS ca_total,
    ROUND(SUM(margin), 0)                               AS marge_totale,
    ROUND(SUM(revenue) / COUNT(DISTINCT order_id), 2)   AS panier_moyen,
    RANK() OVER (ORDER BY SUM(revenue) DESC)            AS rang_ca,
    RANK() OVER (ORDER BY SUM(margin)  DESC)            AS rang_marge
FROM fact_ec
WHERE order_status = 'Livree'
GROUP BY customer_id, pays, segment_client
ORDER BY ca_total DESC
LIMIT 20

### D10 — Clients avec panier moyen élevé (HAVING)

> **MÉTHODE — `HAVING` vs `WHERE` : la distinction la plus importante en SQL analytique.**
>
> L'ordre d'exécution SQL : `FROM` → `WHERE` → `GROUP BY` → **`HAVING`** → `SELECT` → `ORDER BY`
>
> `WHERE` filtre les **lignes** (avant le GROUP BY). `HAVING` filtre les **groupes** (après le GROUP BY). On ne peut pas écrire `WHERE AVG(revenue) > 200` car `AVG` n'existe pas encore quand `WHERE` s'exécute.

In [ ]:
%%sql
SELECT
    customer_id, pays, segment_client,
    COUNT(DISTINCT order_id)    AS nb_commandes,
    ROUND(AVG(revenue), 2)      AS panier_moyen
FROM fact_ec
WHERE order_status = 'Livree'
GROUP BY customer_id, pays, segment_client
HAVING AVG(revenue) > 200
ORDER BY panier_moyen DESC

---
## Bloc E — Canaux d'Acquisition

### E11 — Performance complète par canal avec part de CA ⭐

> **MÉTHODE — `SUM(SUM(revenue)) OVER ()` : la window function imbriquée.**
>
> `SUM(revenue)` à l'intérieur : agrégation du GROUP BY → CA du canal.  
> `SUM(...) OVER ()` à l'extérieur : window function sur **toutes** les lignes → CA global.  
> `OVER ()` vide = pas de partition = total sur toutes les lignes du résultat.  
> Résultat : part de CA en % sans sous-requête séparée.

In [ ]:
%%sql
SELECT
    canal,
    ROUND(SUM(revenue), 0)                                  AS ca_total,
    ROUND(SUM(margin), 0)                                   AS marge_totale,
    ROUND(SUM(margin) / SUM(revenue) * 100, 1)              AS taux_marge_pct,
    COUNT(DISTINCT order_id)                                 AS nb_commandes,
    ROUND(AVG(montant_total), 2)                            AS panier_moyen,
    ROUND(
        SUM(revenue) * 100.0 / SUM(SUM(revenue)) OVER (),
    1)                                                      AS part_ca_pct
FROM fact_ec
WHERE order_status = 'Livree'
GROUP BY canal
ORDER BY ca_total DESC

> **MÉTIER — Ce qu'on dit à M. Diallo :**  
> 'Organic génère X% du CA sans coût variable. Paid Search génère Y% avec le panier moyen le plus élevé. Recommandation : protéger la position SEO et optimiser les campagnes Paid Search en ciblant les catégories à forte marge.'

---
## Bloc F — Risque & Qualité Opérationnelle

### F12 — KPIs de risque globaux

> **MÉTHODE — `CASE WHEN` dans une agrégation : le pattern universel de comptage conditionnel.**
>
> `SUM(CASE WHEN condition THEN 1.0 ELSE 0.0 END)` : transforme une condition en 0/1, puis somme. Équivalent SQL de `.sum()` après un filtre booléen en Python.

In [ ]:
%%sql
SELECT
    ROUND(AVG(CAST(is_risky_order AS DOUBLE)) * 100, 1)      AS taux_risque_pct,
    ROUND(AVG(delivery_delay), 1)                            AS delai_moyen_jours,
    ROUND(
        SUM(CASE WHEN order_status != 'Livree' THEN 1.0 ELSE 0.0 END)
        / COUNT(*) * 100,
    1)                                                       AS pct_non_livree,
    COUNT(DISTINCT CASE WHEN order_status = 'Annulee'
          THEN order_id END)                                 AS nb_annulees,
    COUNT(DISTINCT CASE WHEN order_status = 'Remboursee'
          THEN order_id END)                                 AS nb_remboursees
FROM fact_ec

> **INTERPRÉTATION — Seuils d'alerte :**  
> Taux de risque > 15% : 1 commande sur 7 présente des problèmes.  
> Délai moyen > 7 jours : sous-performant vs standards e-commerce (5 jours cible).  
> % non livrées > 15% : trop de commandes ne finissent pas livrées.

### F13 — Risque par canal

In [ ]:
%%sql
SELECT
    canal,
    COUNT(DISTINCT order_id)                                    AS nb_commandes,
    ROUND(AVG(CAST(is_risky_order AS DOUBLE)) * 100, 1)        AS taux_risque_pct,
    ROUND(AVG(delivery_delay), 1)                              AS delai_moyen_jours
FROM fact_ec
GROUP BY canal
ORDER BY taux_risque_pct DESC

---
## Bloc G — SQL Avancé ⭐⭐

> **MÉTHODE — Ces 3 requêtes couvrent les patterns les plus demandés en entretien SQL.** ROW_NUMBER PARTITION BY, CTEs multiples et CROSS JOIN distinguent un analyste intermédiaire d'un analyste senior.

### G14 — Top 3 catégories par mois (ROW_NUMBER PARTITION BY) ⭐⭐

> **MÉTHODE — `ROW_NUMBER() OVER (PARTITION BY mois ORDER BY ca DESC)` : le pattern le plus puissant du SQL analytique.**
>
> `PARTITION BY mois` : la numérotation **repart à 1** pour chaque mois. Sans PARTITION BY, les rangs continueraient sur tout le résultat.
>
> `WHERE rang_mensuel <= 3` dans la CTE externe : on ne peut pas filtrer sur une window function dans la même CTE. D'où l'encapsulation dans une deuxième CTE.
>
> **Ce pattern — agréger, ranger dans une partition, filtrer sur le rang — résout 80% des problèmes 'top N par groupe' en SQL.**

In [ ]:
%%sql
WITH monthly_cat AS (
    SELECT
        strftime(order_date, '%Y-%m')   AS mois,
        categorie,
        ROUND(SUM(revenue), 0)          AS ca
    FROM fact_ec
    WHERE order_status = 'Livree'
    GROUP BY strftime(order_date, '%Y-%m'), categorie
),
monthly_ranked AS (
    SELECT
        mois, categorie, ca,
        ROW_NUMBER() OVER (PARTITION BY mois ORDER BY ca DESC) AS rang_mensuel
    FROM monthly_cat
)
SELECT *
FROM monthly_ranked
WHERE rang_mensuel <= 3
ORDER BY mois, rang_mensuel

### G15 — Variation mensuelle du CA en % ⭐⭐

In [ ]:
%%sql
WITH monthly AS (
    SELECT
        strftime(order_date, '%Y-%m')   AS mois,
        ROUND(SUM(revenue), 0)          AS ca
    FROM fact_ec
    WHERE order_status = 'Livree'
    GROUP BY strftime(order_date, '%Y-%m')
)
SELECT
    mois, ca,
    LAG(ca) OVER (ORDER BY mois)                            AS ca_mois_precedent,
    ROUND(
        (ca - LAG(ca) OVER (ORDER BY mois))
        / LAG(ca) OVER (ORDER BY mois) * 100,
    1)                                                      AS variation_pct
FROM monthly
ORDER BY mois

### G16 — Clients au-dessus de la moyenne (CTEs multiples + CROSS JOIN) ⭐⭐

> **MÉTHODE — `CROSS JOIN` scalaire : comparer chaque ligne à un agrégat global.**
>
> `moyenne` contient exactement 1 ligne (un seul agrégat global). Le CROSS JOIN ajoute cette valeur à chaque ligne de `client_ca` — pattern standard pour comparer chaque ligne à un total sans sous-requête corrélée.
>
> **Alternative :** `AVG(ca_client) OVER ()` dans la CTE `client_ca` donne le même résultat sans CROSS JOIN. Les deux approches sont correctes.

In [ ]:
%%sql
WITH client_ca AS (
    SELECT
        customer_id, segment_client,
        ROUND(SUM(revenue), 0)  AS ca_client
    FROM fact_ec
    WHERE order_status = 'Livree'
    GROUP BY customer_id, segment_client
),
moyenne AS (
    SELECT AVG(ca_client) AS moy
    FROM client_ca
)
SELECT
    c.customer_id, c.segment_client, c.ca_client,
    ROUND(m.moy, 0)                 AS moyenne_globale,
    ROUND(c.ca_client - m.moy, 0)   AS ecart_moyenne
FROM client_ca c
CROSS JOIN moyenne m
WHERE c.ca_client > m.moy
ORDER BY c.ca_client DESC

> **INTERPRÉTATION :** Si seulement 10% des clients sont au-dessus de la moyenne, la distribution est très concentrée — quelques très gros clients tirent la moyenne vers le haut. L'écart à la moyenne quantifie le 'potentiel de churn' : si le client le plus au-dessus cesse d'acheter, quel est l'impact sur le CA ?

---
### Récapitulatif des patterns SQL maîtrisés

| Pattern SQL | Utilisation | Équivalent Python |
|---|---|---|
| `SUM(col) / SUM(autre_col)` | Taux pondéré (marge) | `df['m'].sum() / df['r'].sum()` |
| `SUM(agrégé) OVER ()` | Total global dans un GROUP BY | `df.groupby()['col'].sum() / total` |
| `LIMIT N` | Limiter les résultats | `df.head(N)` |
| `HAVING condition` | Filtrer après agrégation | `df.groupby().filter()` |
| `LAG(col) OVER (ORDER BY)` | Valeur de la ligne précédente | `df['col'].shift(1)` |
| `RANK() OVER (ORDER BY)` | Rang sans suppression de lignes | `df['col'].rank()` |
| `ROW_NUMBER() OVER (PARTITION BY)` | Top N par groupe | `df.groupby().head(N)` |
| `WITH cte1 AS (...), cte2 AS (...)` | Requête en étapes | Variables intermédiaires |
| `CROSS JOIN table_1_ligne` | Comparer à un agrégat global | `merge` sur une constante |

---
# PARTIE 2 — Segmentation RFM en Python

> **MÉTHODE — Pourquoi Python pour la segmentation et non SQL pur ?**
>
> La segmentation RFM nécessite des opérations que Python gère mieux : découpage en quantiles égaux (`pd.qcut`) avec gestion des doublons, règles métier multi-colonnes (`apply`), visualisation avancée.

---
## Étape 1 — Préparation du dataset

> **MÉTHODE — Filtrer sur `order_status = 'Livree'` pour la RFM.**  
> Les commandes annulées et remboursées ne doivent pas contribuer aux métriques RFM. Un client qui a commandé puis annulé n'a pas vraiment 'dépensé' ce montant.

In [ ]:
df_liv = fact_ec[fact_ec["order_status"] == "Livree"].copy()

print(f"Dataset total    : {len(fact_ec):,} lignes")
print(f"Commandes livrées : {len(df_liv):,} lignes")
print(f"Période : {df_liv['order_date'].min().date()} → {df_liv['order_date'].max().date()}")
print(f"Clients uniques  : {df_liv['customer_id'].nunique():,}")

---
## Étape 2 — Calcul des métriques RFM

| Métrique | Définition | Calcul | Sens du score |
|---|---|---|---|
| **R** — Recency | Jours depuis la dernière commande | `date_ref - max(order_date)` | Petit = récent = BIEN |
| **F** — Frequency | Nombre de commandes distinctes | `COUNT(DISTINCT order_id)` | Grand = fréquent = BIEN |
| **M** — Monetary | CA total dépensé | `SUM(revenue)` | Grand = dépensier = BIEN |

> **MÉTHODE — Date de référence fixe : `max(order_date) + 1 jour`.**  
> Si on utilise `pd.Timestamp.now()`, la Recency de tous les clients augmente chaque jour. Un notebook exécuté dans 6 mois produira des résultats différents. Avec une date fixe ancrée sur le dataset : résultats reproductibles et auditables.
>
> **`nunique()` pour Frequency :** un client avec 3 produits dans 1 commande a 3 lignes. `nunique()` retourne 1 commande unique. La fréquence compte les commandes, pas les lignes produits.

In [ ]:
DATE_REF = df_liv["order_date"].max() + pd.Timedelta(days=1)
print(f"Date de référence : {DATE_REF.date()}")

rfm = (
    df_liv.groupby("customer_id")
    .agg(
        last_order = ("order_date",     "max"),
        frequency  = ("order_id",       "nunique"),
        monetary   = ("revenue",        "sum"),
        segment    = ("segment_client", "first"),
        pays       = ("pays",           "first")
    )
    .reset_index()
)

rfm["recency"] = (DATE_REF - rfm["last_order"]).dt.days
rfm = rfm.drop(columns="last_order")

print(f"\nDataset RFM : {len(rfm):,} clients")
print(f"  Recency   — moy : {rfm['recency'].mean():.0f}j  | médiane : {rfm['recency'].median():.0f}j")
print(f"  Frequency — moy : {rfm['frequency'].mean():.1f}   | médiane : {rfm['frequency'].median():.0f}")
print(f"  Monetary  — moy : {rfm['monetary'].mean():.0f}€  | médiane : {rfm['monetary'].median():.0f}€")

rfm.head(5)

---
## Étape 3 — Calcul des scores R, F, M

> **MÉTHODE — `pd.qcut` : découpage en quantiles de taille égale.**
>
> `pd.qcut(serie, q=5, labels=[1,2,3,4,5])` divise les valeurs en 5 groupes contenant **exactement le même nombre** de clients.
>
> **Pourquoi `rank(method='first')` avant `pd.qcut` pour F et M ?**  
> `pd.qcut` échoue s'il y a trop de valeurs identiques aux frontières des quantiles (ex : 30% des clients ont frequency=1). `rank(method='first')` transforme les valeurs en rangs uniques, permettant à `qcut` de diviser proprement.
>
> **Pourquoi les labels `[5,4,3,2,1]` pour R (inversé) ?**  
> Pour la Recency, une valeur faible = client récent = bon. En inversant les labels, le quintile le plus bas (clients les plus récents) reçoit le score 5.

In [ ]:
# Score R : inversé (récent = bon = score 5)
rfm["R_score"] = pd.qcut(
    rfm["recency"],
    q=5,
    labels=[5, 4, 3, 2, 1]   # inversion : petite recency = score élevé
).astype(int)

# Score F : rank avant qcut pour éviter les erreurs sur les doublons
rfm["F_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Score M : même logique que F
rfm["M_score"] = pd.qcut(
    rfm["monetary"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Score RFM concaténé (ex : "435") et total (de 3 à 15)
rfm["rfm_score"] = rfm["R_score"].astype(str) + rfm["F_score"].astype(str) + rfm["M_score"].astype(str)
rfm["rfm_total"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

print("Distribution des scores (doit être ~uniforme, ~20% par score) :")
for col in ["R_score", "F_score", "M_score"]:
    dist = rfm[col].value_counts().sort_index()
    pcts = (dist / len(rfm) * 100).round(1)
    print(f"  {col}: " + " | ".join(f"{k}:{pcts[k]}%" for k in pcts.index))

rfm[["customer_id", "recency", "frequency", "monetary",
     "R_score", "F_score", "M_score", "rfm_score", "rfm_total"]].head(8)

> **INTERPRÉTATION — Le `rfm_score` concaténé :**  
> `'555'` = Champion parfait. `'511'` = Nouveau client récent, pas encore fréquent. `'115'` = Dormant qui a dépensé beaucoup historiquement. Ces codes sont utilisables dans les outils CRM pour des règles d'automatisation.
>
> **Validation de la distribution :** si un score est surreprésenté (> 25%), utiliser `rank(method='first')` ou réduire à 4 quantiles.

---
## Étape 4 — Segmentation métier

> **MÉTHODE — La segmentation RFM est une décision métier, pas technique.**  
> Les règles ci-dessous sont un point de départ à valider avec M. Diallo et les équipes marketing.
>
> **`apply(axis=1)` : appliquer une fonction sur chaque ligne.**  
> `axis=1` = 'pour chaque ligne'. La fonction reçoit une Series représentant toute la ligne et peut accéder à R, F, M simultanément. Pattern standard pour les règles métier multi-colonnes.

In [ ]:
def segment_rfm(row):
    r, f, m = row["R_score"], row["F_score"], row["M_score"]
    total   = row["rfm_total"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Fidèles"
    elif m >= 4 and f <= 2:
        return "Gros dépensiers occasionnels"
    elif r <= 2 and total >= 8:
        return "À réactiver"
    elif r >= 4 and f <= 2:
        return "Nouveaux prometteurs"
    else:
        return "Dormants"

rfm["segment_rfm"] = rfm.apply(segment_rfm, axis=1)

dist = rfm["segment_rfm"].value_counts().reset_index()
dist.columns = ["segment", "nb_clients"]
dist["pct"] = (dist["nb_clients"] / dist["nb_clients"].sum() * 100).round(1)

print("DISTRIBUTION DE LA SEGMENTATION RFM :")
display(dist)

print(f"\n  Champions + Fidèles (à chérir)  : {dist[dist['segment'].isin(['Champions','Fidèles'])]['nb_clients'].sum():,} clients")
print(f"  À réactiver (à récupérer)       : {dist[dist['segment']=='À réactiver']['nb_clients'].sum():,} clients")
print(f"  Dormants (à décision)           : {dist[dist['segment']=='Dormants']['nb_clients'].sum():,} clients")

> **INTERPRÉTATION — La distribution des segments :**
>
> **Si Champions + Fidèles > 30% :** la base client est saine avec un noyau dur d'acheteurs récurrents.
>
> **Si Dormants > 40% :** signal d'alerte — la majorité des clients inscrits ont une faible activité. Le coût d'acquisition est mal rentabilisé.
>
> **MÉTIER — Le ROI de la segmentation RFM :**  
> Sans segmentation : 1 email à 3000 clients, conversion 2% = 60 conversions.  
> Avec segmentation : Champions (200 clients, 15%) + À réactiver (400 clients, 8%) + Nouveaux (300 clients, 10%) = 92 conversions sur 900 emails. **+53% de conversions avec 70% d'emails en moins.**

---
## Étape 5 — Visualisation

> **MÉTHODE — 3 graphiques complémentaires pour 3 angles d'analyse.**  
> **Donut :** combien de clients par segment ?  
> **Barres CA :** quel segment génère le plus de CA ?  
> **Scatter R vs M :** comment les segments se positionnent dans l'espace ?

In [ ]:
colors_seg = {
    "Champions":                    COLORS["secondary"],
    "Fidèles":                      COLORS["primary"],
    "Gros dépensiers occasionnels": COLORS["warning"],
    "Nouveaux prometteurs":         "#5DCAA5",
    "À réactiver":                  COLORS["danger"],
    "Dormants":                     COLORS["neutral"],
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Donut : répartition des segments
dist_plot   = dist.set_index("segment")
plot_colors = [colors_seg.get(s, COLORS["neutral"]) for s in dist_plot.index]
axes[0].pie(
    dist_plot["nb_clients"],
    labels=dist_plot.index, colors=plot_colors,
    autopct="%1.1f%%", wedgeprops=dict(edgecolor="white", linewidth=2),
    startangle=90, pctdistance=0.75, labeldistance=1.12, textprops={"fontsize": 9}
)
axes[0].add_patch(plt.Circle((0, 0), 0.55, color="white"))
axes[0].set_title("Répartition des segments RFM\n(% des clients)", fontweight="bold")

# CA total par segment
ca_seg = (
    rfm.merge(df_liv.groupby("customer_id")["revenue"].sum().reset_index(), on="customer_id", how="left")
    .groupby("segment_rfm")["revenue"].sum()
    .sort_values(ascending=False)
)
ca_seg_pct = ca_seg / ca_seg.sum() * 100

seg_colors = [colors_seg.get(s, COLORS["neutral"]) for s in ca_seg.index]
bars = axes[1].bar(ca_seg.index, ca_seg.values / 1000, color=seg_colors, alpha=0.85, edgecolor="white")
for bar, v, pct in zip(bars, ca_seg.values / 1000, ca_seg_pct.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{pct:.1f}%", ha="center", fontsize=8, fontweight="bold")
axes[1].set_title("CA total par segment\n(% du CA global annoté)", fontweight="bold")
axes[1].tick_params(axis="x", rotation=25)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}k"))
axes[1].set_ylabel("CA (k€)")

# Scatter Recency vs Monetary
for seg, grp in rfm.groupby("segment_rfm"):
    axes[2].scatter(
        grp["recency"], grp["monetary"] / 1000,
        alpha=0.55, s=25, label=seg, color=colors_seg.get(seg, COLORS["neutral"])
    )
axes[2].set_xlabel("Recency (jours depuis dernière commande)")
axes[2].set_ylabel("CA client (k€)")
axes[2].set_title("Recency vs Monetary\n(position des segments)", fontweight="bold")
axes[2].legend(fontsize=8, markerscale=1.5)

plt.suptitle("Segmentation RFM — ShopAfrica+", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("rfm_segmentation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé : rfm_segmentation.png")

> **INTERPRÉTATION — Scatter Recency vs Monetary :**  
> Champions : en bas à gauche (récents ET fort CA).  
> À réactiver : en bas à droite (fort CA historique mais recency élevée — ce sont les clients à récupérer en priorité).  
> Dormants : dispersés avec peu de CA et pas récents.

---
## Étape 6 — Statistiques par segment

> **MÉTHODE — Les statistiques moyennes par segment permettent de vérifier que les segments sont bien différenciés.** Si les stats de deux segments sont très proches, ajuster les seuils dans `segment_rfm()`.

In [ ]:
stats_seg = (
    rfm.groupby("segment_rfm")
    .agg(
        nb_clients    = ("customer_id", "count"),
        recency_moy   = ("recency",     "mean"),
        frequency_moy = ("frequency",   "mean"),
        monetary_moy  = ("monetary",    "mean"),
        r_moy         = ("R_score",     "mean"),
        f_moy         = ("F_score",     "mean"),
        m_moy         = ("M_score",     "mean")
    )
    .round(1)
    .sort_values("monetary_moy", ascending=False)
    .reset_index()
)
display(stats_seg)

# Contribution au CA par segment
ca_par_seg = (
    rfm
    .merge(df_liv.groupby("customer_id")["revenue"].sum().reset_index(), on="customer_id", how="left")
    .groupby("segment_rfm")["revenue"]
    .agg(["sum", "mean"]).round(0)
)
ca_par_seg.columns = ["CA total", "CA moyen/client"]
ca_par_seg["% du CA total"] = (ca_par_seg["CA total"] / ca_par_seg["CA total"].sum() * 100).round(1)
print("\nContribution au CA par segment :")
display(ca_par_seg.sort_values("CA total", ascending=False))

---
## Étape 7 — Recommandations business par segment

| Segment | Qui sont-ils ? | Action prioritaire | KPI de succès |
|---|---|---|---|
| **Champions** | Récents, fréquents, gros dépensiers | Programme VIP, early access, referral | Rétention > 90% |
| **Fidèles** | Fréquents mais panier modéré | Upsell, cross-sell, offres premium | Hausse panier moyen |
| **Gros dépensiers occasionnels** | Fort panier, faible fréquence | Incentive fréquence, abonnement | +1 commande/an |
| **Nouveaux prometteurs** | 1er achat récent, potentiel | Welcome journey, 2ème achat dans 30j | Taux 2ème achat |
| **À réactiver** | Inactifs mais valeur historique | Email de réactivation + offre de retour | Taux de réactivation |
| **Dormants** | Faibles sur les 3 dimensions | CRM allégé, désabonnement | — |

---
## Étape 8 — Export du dataset RFM

In [ ]:
# Export CSV → importé dans Power BI (étape 4)
rfm.to_csv("clients_rfm_segments.csv", index=False)
print(f"Exporté : clients_rfm_segments.csv ({len(rfm):,} clients) ✅")
print(f"Colonnes : {rfm.columns.tolist()}")

rfm.head(10)

---
## Checklist de complétion

**Partie SQL — DuckDB (`%%sql`) :**
- [x] Bloc A — KPIs globaux (CA, marge, panier moyen en 2 étapes, clients actifs)
- [x] Bloc B — Analyse temporelle (strftime, LIMIT 3, anomalies avec LAG + CTE)
- [x] Bloc C — Catégories (RANK OVER pour double classement, top marques)
- [x] Bloc D — Clients (top 20, HAVING vs WHERE)
- [x] Bloc E — Canaux (SUM(SUM()) OVER pour part CA en %)
- [x] Bloc F — Risque (CASE WHEN conditionnel, CAST AS DOUBLE)
- [x] Bloc G — SQL avancé (ROW_NUMBER PARTITION BY, LAG, CTEs multiples, CROSS JOIN)

**Partie Python — RFM :**
- [x] Date de référence fixe (reproductibilité)
- [x] Métriques RFM calculées avec `nunique()` pour frequency
- [x] Scores via `pd.qcut` avec `rank(method='first')` pour les doublons
- [x] Labels inversés `[5,4,3,2,1]` pour Recency
- [x] Segmentation métier avec `apply(axis=1)`
- [x] 3 visualisations (donut, CA, scatter R vs M)
- [x] Statistiques par segment avec ratio valeur/volume
- [x] Recommandations actionnables par segment avec KPIs de succès
- [x] Export `clients_rfm_segments.csv`

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.